# Free Colab pilot: Qwen teacher → native Ron student

This notebook is designed for a **free Google Colab runtime**. Free GPU availability is not guaranteed; this notebook never purchases compute or calls a paid inference API.

It performs a small pilot, not a production training run:
1. Load Qwen2.5-7B-Instruct in 4-bit on a CUDA GPU.
2. Generate a small set of teacher answers.
3. Require you to inspect answers and explicitly approve selected rows.
4. Train Ron's own BPE tokenizer and native scalable Transformer with supervised fine-tuning.
5. Save the student checkpoint and metrics to Google Drive.

Open in Colab: https://colab.research.google.com/github/arkanws513-spec/Ron-0/blob/feat/qwen25-7b-distillation-clean/notebooks/free_colab_distillation.ipynb

**Important:** Keep this notebook on a GPU runtime. If Colab does not provide a free GPU, stop and try again later rather than enabling paid compute. The 7B teacher download can be several GB and the session can disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone --depth 1 --branch feat/qwen25-7b-distillation-clean https://github.com/arkanws513-spec/Ron-0.git /content/Ron-0
%cd /content/Ron-0

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No free CUDA GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU if available, or try again later. This notebook will not use paid compute.")
print("GPU:", torch.cuda.get_device_name(0))

!pip -q install -e ".[distill]" bitsandbytes tokenizers

## 1. Generate teacher examples

The initial repository prompt set is small (about 30 prompts) so this is a low-cost smoke/pilot run. For a more useful dataset, add a reviewed, legally usable prompt file and pass it with `--prompts`. Outputs are **not approved automatically**.

In [ ]:
!python scripts/distill_qwen25.py --load-in-4bit --limit 30 --max-new-tokens 192 --output-jsonl /content/teacher.jsonl --output-corpus /content/teacher_corpus.txt

## 2. Review the teacher's answers

Read every sample below. Do not approve inaccurate, unsafe, low-quality, or duplicated responses. Edit `APPROVED_INDICES` in the next cell to include only the 1-based row numbers you personally reviewed. The default is empty on purpose.

In [ ]:
import json
from pathlib import Path

teacher_path = Path("/content/teacher.jsonl")
teacher_rows = [json.loads(line) for line in teacher_path.read_text(encoding="utf-8").splitlines() if line.strip()]
for i, row in enumerate(teacher_rows, 1):
    print(f"\n{'='*90}\nROW {i}\nUSER: {row['prompt']}\n\nTEACHER: {row['response']}")

# IMPORTANT: manually replace [] with the 1-based indices you inspected and approve.
APPROVED_INDICES = []
print(f"Loaded {len(teacher_rows)} examples. Approved so far: {len(APPROVED_INDICES)}")

In [ ]:
approved = set(APPROVED_INDICES)
if not approved:
    raise RuntimeError("No rows approved. Review the output above, edit APPROVED_INDICES, then rerun this cell.")
if any(i < 1 or i > len(teacher_rows) for i in approved):
    raise ValueError("APPROVED_INDICES contains an out-of-range row number.")

reviewed_rows = []
for i, row in enumerate(teacher_rows, 1):
    if i in approved:
        row = dict(row)
        row["review_status"] = "approved"
        reviewed_rows.append(row)

work = Path("/content/ron_pilot")
work.mkdir(parents=True, exist_ok=True)
data_path = work / "reviewed_teacher.jsonl"
data_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in reviewed_rows), encoding="utf-8")
corpus_path = work / "reviewed_corpus.txt"
corpus_path.write_text("\n\n".join(f"User: {r['prompt']}\nRon: {r['response']}" for r in reviewed_rows) + "\n", encoding="utf-8")
print(f"Approved {len(reviewed_rows)} records. Files saved under {work}")

## 3. Train Ron's tokenizer and student model

This creates a **new native Ron checkpoint** from the reviewed examples. The small dataset is only a smoke test; it does not establish broad reasoning ability. The student runs independently at inference time and does not load Qwen's weights.

In [ ]:
!python scripts/train_ron_tokenizer.py --corpus /content/ron_pilot/reviewed_corpus.txt --output /content/ron_pilot/tokenizer --vocab-size 8000
!python scripts/train_scalable_sft.py --data /content/ron_pilot/reviewed_teacher.jsonl --tokenizer /content/ron_pilot/tokenizer/tokenizer.json --output /content/ron_pilot/student --hidden-size 256 --layers 4 --heads 4 --kv-heads 2 --sequence-length 256 --epochs 2 --batch-size 2 --learning-rate 0.0003

## 4. Save the result to Google Drive

The artifacts are copied to Drive so they survive the temporary Colab session. Keep the checkpoint private if you do not want to publish it. Do not commit model weights to Git.

In [ ]:
import shutil, json
from pathlib import Path

destination = Path("/content/drive/MyDrive/Ron-0-distillation-pilot")
destination.mkdir(parents=True, exist_ok=True)
shutil.copytree("/content/ron_pilot/student", destination / "student", dirs_exist_ok=True)
shutil.copy2("/content/ron_pilot/reviewed_teacher.jsonl", destination / "reviewed_teacher.jsonl")
shutil.copy2("/content/ron_pilot/reviewed_corpus.txt", destination / "reviewed_corpus.txt")
print("Saved pilot artifacts to:", destination)
metrics = json.loads((destination / "student" / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))